# 0. import

In [1]:
import sys
import os, time, torch, tqdm, numpy as np, pandas as pd


from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import HeteroConv, SAGEConv
from torch.amp import autocast, GradScaler         
from torch.nn.utils import clip_grad_norm_

from torch_geometric.utils import negative_sampling

from neo4j.exceptions import ServiceUnavailable
from neomodel import config as neomodel_config
from neomodel import db

from app.config.local import NEO4J_HOST, NEO4J_PORT, NEO4J_USER, DB_PASSWORD
NEO4J_USER     = NEO4J_USER
NEO4J_PWD      = DB_PASSWORD
NEO4J_HOST     = NEO4J_HOST
NEO4J_PORT     = NEO4J_PORT

neomodel_config.DATABASE_URL = f'bolt://{NEO4J_USER}:{NEO4J_PWD}@{NEO4J_HOST}:{NEO4J_PORT}'

try:
    result, _ = db.cypher_query("RETURN 1 AS test")
    print("✅ Neo4j 연결 성공! → 응답값:", result[0][0])
except ServiceUnavailable as e:
    print("❌ Neo4j 연결 실패:", e)

✅ config_module: ['CARD_SERVER_ADDRESS', 'DB_HOST', 'DB_NAME', 'DB_PASSWORD', 'DB_PORT', 'DB_USER', 'DEBUG', 'GEMINI_API_KEY', 'GOOGLE_APPLICATION_CREDENTIALS', 'GOOGLE_PROJECT', 'NEO4J_HOST', 'NEO4J_PORT', 'NEO4J_USER', 'SWAGGER_SPECS_JSON_ROUTE', 'SWAGGER_SPECS_ROUTE', 'SWAGGER_STATIC_PATH', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'os']
✅ Neo4j 연결 성공! → 응답값: 1


### 기초 함수 정의

In [2]:
# 단계별 경과 시간을 콘솔에 출력.
def log(step: str, start: float):
    dur = time.perf_counter() - start
    print(f"[{step}] ✅ 완료  |  {dur:6.2f} s")
START_ALL = time.perf_counter()

# conver cypher to df(pandas)
def cypher_df(q: str, params: dict | None = None) -> pd.DataFrame:
    records, cols = db.cypher_query(q, params or {})
    df = pd.DataFrame(records, columns=cols)

    # 1) 어떤 컬럼을 변환할지 결정
    id_cols = [c for c in df.columns
               if c.endswith('_id') or c in {'mid', 'oid', 'pid'}]

    # 2) 캐스팅 (문자·공백·NULL → NaN → Int64 → int64)
    for col in id_cols:
        df[col] = (
            pd.to_numeric(df[col], errors="coerce")  # 문자열·공백 처리
              .astype("Int64")                      # Nullable 정수
              .astype("int64")                      # 확정 int64
        )
    return df

# enable safe mapping(drop rows when failure)
def safe_map(series: pd.Series, mapping: dict[int,int], col: str) -> np.ndarray:
    mapped = series.map(mapping)
    miss   = mapped.isna().sum()
    if miss:
        print(f"⚠️  {col}: {miss:,}개 매핑 실패 → 행 제거")
        mapped = mapped.dropna()
    return mapped.astype(np.int64).to_numpy()



records, _ = db.cypher_query("""
MATCH (m:Member)
RETURN m.member_id AS mid
ORDER BY mid
""")
mids = [mid for (mid,) in records]        

# mids를 int형으로 변환 후 numpy array로
mids_arr = np.array([int(mid) for mid in mids])

min_id, max_id = mids_arr.min(), mids_arr.max()
missing = set(range(min_id, max_id + 1)) - set(mids_arr)
is_serialized = len(missing) == 0 and (np.diff(np.sort(mids_arr)) == 1).all()

print("min_id:", min_id)
print("max_id:", max_id)
print("missing:", sorted(missing))
print("직렬화 여부:", is_serialized)


# Neo4j에서 order_id 목록 가져오기
records, _ = db.cypher_query("MATCH (o:Order) RETURN o.order_id")
order_ids = [int(oid) for (oid,) in records]

# numpy로 변환 후 직렬화 여부 검사
import numpy as np
order_arr = np.array(order_ids)
min_oid, max_oid = order_arr.min(), order_arr.max()
missing_order = set(range(min_oid, max_oid + 1)) - set(order_arr)
is_serialized_order = len(missing_order) == 0 and (np.diff(np.sort(order_arr)) == 1).all()

print("🧾 [Order] ID 범위:", min_oid, "~", max_oid)
print("⛔ 빠진 Order ID:", sorted(missing_order))
print("✅ 직렬화 여부 (Order):", is_serialized_order)

### 이거 없는 이유 : eval_set = test인 주문을 삭제

records, _ = db.cypher_query("MATCH (p:Product) RETURN p.product_id")
product_ids = [int(pid) for (pid,) in records]

product_arr = np.array(product_ids)
min_pid, max_pid = product_arr.min(), product_arr.max()
missing_product = set(range(min_pid, max_pid + 1)) - set(product_arr)
is_serialized_product = len(missing_product) == 0 and (np.diff(np.sort(product_arr)) == 1).all()

print("🛒 [Product] ID 범위:", min_pid, "~", max_pid)
print("⛔ 빠진 Product ID:", sorted(missing_product))
print("✅ 직렬화 여부 (Product):", is_serialized_product)

### 이거 없는 이유 : 애초에 주문 내역이 없어서 병합 과정에서 삭제됨

## 데이터 불러오기

### 데이터 직렬화(인덱스 정렬)

In [3]:
t0 = time.perf_counter()

# 1) 노드-ID 추출 ────────────────
df_member  = cypher_df("MATCH (m:Member)  RETURN m.member_id  AS mid")
df_order   = cypher_df("MATCH (o:Order)   RETURN o.order_id   AS oid")
df_product = cypher_df("MATCH (p:Product) RETURN p.product_id AS pid")

# 2) dict ↔ index 매핑 ──────────
m2i = {mid: i for i, mid in enumerate(df_member.mid.unique())}
o2i = {oid: i for i, oid in enumerate(df_order.oid.unique())}
p2i = {pid: i for i, pid in enumerate(df_product.pid.unique())}

# 역매핑: NumPy 배열로  O(1) 룩업
i2m = np.fromiter(m2i.keys(), dtype=np.int64)
i2o = np.fromiter(o2i.keys(), dtype=np.int64)
i2p = np.fromiter(p2i.keys(), dtype=np.int64)

print(f"users={len(m2i):,} | orders={len(o2i):,} | products={len(p2i):,}")
log("ID→Index 매핑", t0)

users=206,209 | orders=3,346,083 | products=49,685
[ID→Index 매핑] ✅ 완료  |   76.76 s


### 피쳐 로드

In [4]:
t0 = time.perf_counter()

# 2-1. Order feature  ────────────────────────────
df_order_feat = cypher_df("""
MATCH (o:Order)
RETURN o.order_id AS oid,
       o.days_since_prior_order_norm AS dsp,
       o.order_dow_norm              AS dow,
       o.order_hour_of_day_norm      AS hr,
       o.order_number_norm           AS idx
""")

# ※ 매핑 실패 행 제거 (oid → o2i)
df_order_feat["oid_idx"] = safe_map(df_order_feat.oid, o2i, "oid")
order_feat = (
    df_order_feat
      .sort_values("oid_idx")[["dsp","dow","hr","idx"]]
      .to_numpy(np.float32)
)                              

# 2-2. Product feature  ──────────────────────────
df_prod_vec = cypher_df("""
MATCH (p:Product)
RETURN p.product_id AS pid,
       p.category_vector + p.name_vector AS vec
""")

df_prod_vec["pid_idx"] = safe_map(df_prod_vec.pid, p2i, "pid")
prod_feat = torch.tensor(
    np.stack(df_prod_vec.sort_values("pid_idx").vec.values),
    dtype=torch.float32
)                                             # shape = [N_prod, D]
prod_feat_dim = prod_feat.size(1)

print(f"Order feature  : {order_feat.shape}")
print(f"Product vector : {prod_feat.shape}")
log("피처 로딩", t0)


Order feature  : (3346083, 4)
Product vector : torch.Size([49685, 2048])
[피처 로딩] ✅ 완료  |  154.39 s


### 학습용 서브그래프

In [5]:
t0 = time.perf_counter()

# 3-1. prior Edge  ────────────────────────────────────────────────
df_edge_tr = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE o.role_train = 'prior'
RETURN m.member_id AS mid, o.order_id AS oid, p.product_id AS pid
""")

uo = torch.stack((
    torch.from_numpy(safe_map(df_edge_tr.mid, m2i, "mid")),
    torch.from_numpy(safe_map(df_edge_tr.oid, o2i, "oid"))
), 0).to(torch.long)                            # [2, E_tr]

op = torch.stack((
    torch.from_numpy(safe_map(df_edge_tr.oid, o2i, "oid")),
    torch.from_numpy(safe_map(df_edge_tr.pid, p2i, "pid"))
), 0).to(torch.long)                            # [2, E_tr]

print(f"Train-prior edges : {uo.size(1):,}")

# Positive label
df_label_tr = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE o.role_train = 'train'
RETURN m.member_id AS mid, p.product_id AS pid
""")

pos_u = torch.from_numpy(safe_map(df_label_tr.mid, m2i, "mid"))
pos_p = torch.from_numpy(safe_map(df_label_tr.pid, p2i, "pid"))
edge_pos = torch.stack((pos_u, pos_p), 0).to(torch.long)

print(f"Positive pairs    : {edge_pos.size(1):,}")

# Negative sampling (유저당 10개)
from torch_geometric.utils import negative_sampling
NEG_K   = 10
num_neg = edge_pos.size(1) * NEG_K

edge_neg = negative_sampling(
    edge_index=edge_pos,
    num_nodes=(len(m2i), len(p2i)),
    num_neg_samples=num_neg            #
).to(torch.long)

print(f"Negative samples  : {edge_neg.size(1):,}")

# 3-4. edge_label_index & edge_label
edge_label_index = torch.cat([edge_pos, edge_neg], 1)    
edge_label = torch.cat([
    torch.ones(edge_pos.size(1)),
    torch.zeros(edge_neg.size(1))
])

log("서브그래프 구성", t0)

Train-prior edges : 30,883,423
Positive pairs    : 770,266
Negative samples  : 7,702,660
[서브그래프 구성] ✅ 완료  |  726.06 s


In [6]:
# ── HeteroData ─────────────────────────────────────────────────────────
data = HeteroData()
data['user'].x    = nn.Embedding(len(m2i), 32).weight          # 32-dim 임베딩
data['order'].x   = order_feat                                 # 4-dim 실수
data['product'].x = prod_feat                                  # D-dim 실수
data['user'].num_nodes, data['order'].num_nodes, data['product'].num_nodes = \
    len(m2i), len(o2i), len(p2i)

data['user','ordered','order'].edge_index       = uo
data['order','ordered_by','user'].edge_index    = uo.flip(0)
data['order','contains','product'].edge_index   = op
data['product','contained_in','order'].edge_index = op.flip(0)

### 테스트용 서브그래프 

In [7]:
# 1. prior edge  (role_test = 'prior')
t0 = time.perf_counter()

df_edge_te = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE o.eval_set='test' AND o.role_test='prior'
RETURN m.member_id AS mid, o.order_id AS oid, p.product_id AS pid
""")

uo_te = torch.stack((
    torch.from_numpy(safe_map(df_edge_te.mid, m2i, "mid")),
    torch.from_numpy(safe_map(df_edge_te.oid, o2i, "oid"))
), 0).to(torch.long)

op_te = torch.stack((
    torch.from_numpy(safe_map(df_edge_te.oid, o2i, "oid")),
    torch.from_numpy(safe_map(df_edge_te.pid, p2i, "pid"))
), 0).to(torch.long)

print(f"Test-prior edges   : {uo_te.size(1):,}")

# 2. positive label  (role_test = 'train')
df_label_te = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE o.eval_set='test' AND o.role_test='train'
RETURN m.member_id AS mid, p.product_id AS pid
""")

pos_u = torch.from_numpy(safe_map(df_label_te.mid, m2i, "mid"))
pos_p = torch.from_numpy(safe_map(df_label_te.pid, p2i, "pid"))
edge_pos_te = torch.stack((pos_u, pos_p), 0).to(torch.long)

print(f"Positive pairs     : {edge_pos_te.size(1):,}")


# 3. negative sampling  (유저당 10개)
from torch_geometric.utils import negative_sampling
NEG_K  = 10
edge_neg_te = negative_sampling(
    edge_index=edge_pos_te,
    num_nodes=(len(m2i), len(p2i)),
    num_neg_samples=edge_pos_te.size(1) * NEG_K
).to(torch.long)

print(f"Negative samples   : {edge_neg_te.size(1):,}")


# 4. edge_label_index & edge_label
edge_label_index_te = torch.cat([edge_pos_te, edge_neg_te], 1)
edge_label_te = torch.cat([
    torch.ones(edge_pos_te.size(1)),
    torch.zeros(edge_neg_te.size(1))
])

log("테스트 서브그래프", t0)

# ───────────────────────────────────────────────────────────
# 5. HeteroData (테스트용)
# ───────────────────────────────────────────────────────────
test_data = HeteroData()
test_data['user'].num_nodes, test_data['order'].num_nodes, test_data['product'].num_nodes = \
    len(m2i), len(o2i), len(p2i)

# feature 재활용 (학습 때 만든 order_feat, prod_feat 사용)
test_data['user'].x    = nn.Embedding(len(m2i), 32).weight      # 임베딩 초기화
test_data['order'].x   = torch.tensor(order_feat, dtype=torch.float32)
test_data['product'].x = prod_feat

# edge 등록 (정방향 + 역방향)
test_data['user','ordered','order'].edge_index        = uo_te
test_data['order','ordered_by','user'].edge_index     = uo_te.flip(0)
test_data['order','contains','product'].edge_index    = op_te
test_data['product','contained_in','order'].edge_index= op_te.flip(0)


Test-prior edges   : 11,011,698
Positive pairs     : 780,800
Negative samples   : 7,808,000
[테스트 서브그래프] ✅ 완료  |  297.71 s


# 2. 모델 정의

In [8]:
# ───────────────────────────────────────────────────────────
# 1. HetSAGE 모델 정의
# ───────────────────────────────────────────────────────────
class HetSAGE(nn.Module):
    def __init__(self, hid: int = 64, out: int = 64):
        super().__init__()
        # ① 1-hop
        self.conv1 = HeteroConv({
            ('user',   'ordered',      'order'):   SAGEConv((-1, -1), hid),
            ('order',  'contains',     'product'): SAGEConv((-1, -1), hid),
            ('order',  'ordered_by',   'user'):    SAGEConv((-1, -1), hid),  # 역
            ('product','contained_in', 'order'):   SAGEConv((-1, -1), hid)   # 역
        }, aggr='mean')
        # ② 2-hop
        self.conv2 = HeteroConv({
            ('user',   'ordered',      'order'):   SAGEConv((hid, hid), out),
            ('order',  'contains',     'product'): SAGEConv((hid, hid), out),
            ('order',  'ordered_by',   'user'):    SAGEConv((hid, hid), out),
            ('product','contained_in', 'order'):   SAGEConv((hid, hid), out)
        }, aggr='mean')

    def forward(self, x_dict, edge_dict):
        h = {k: torch.relu(v) for k, v in self.conv1(x_dict, edge_dict).items()}
        return self.conv2(h, edge_dict)                 # h_dict ⟶ 최종 임베딩

In [9]:

# ───────────────────────────────────────────────────────────
# 2. 학습용 HeteroData에 label-edge 타입 등록
#    (LinkNeighborLoader 가 요구)
# ───────────────────────────────────────────────────────────
# edge_label_index = [2, P+N]  ← 3-단계에서 이미 생성
data['user', 'buys', 'product'].edge_index = edge_label_index   # ★ dummy 등록


# ───────────────────────────────────────────────────────────
# 3. LinkNeighborLoader
# ───────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = HetSAGE().to(device)

loader = LinkNeighborLoader(
    data,
    edge_label_index=(('user','buys','product'), edge_label_index),
    edge_label=edge_label,
    batch_size=1024,         
    num_neighbors=[15, 5],   
    shuffle=True
)

print("Mini-batches :", len(loader))


Mini-batches : 8275


# 학습 전에 더 이상 사용하지 않는 메모리 해제

In [10]:
# ───────────────────────────────────────────────────────────
# 메모리 정리: 학습에 불필요한 객체 즉시 해제
#   · DataFrame 원본          df_*
#   · 임시 배열 / 텐서        *_tmp, edge_neg 등
#   · (필요 시) CPU→GPU 전송 후의 소스 텐서
# ───────────────────────────────────────────────────────────
import gc, torch

def free_mem(*names):
    """글로벌 네임스페이스에서 변수 삭제 → GC → CUDA 캐시 비우기"""
    g = globals()
    for n in names:
        if n in g:
            del g[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── 1. 노드 ID DataFrame (더 이상 안 씀)
free_mem('df_member', 'df_order', 'df_product')

# ── 2. 피처 생성에만 쓰인 DF
free_mem('df_order_feat', 'df_prod_vec')

# ── 3. 학습용 edge/label 원본 DF
free_mem('df_edge_tr', 'df_label_tr')

print("🗑️  불필요 객체 삭제 및 CUDA 캐시 정리 완료")

🗑️  불필요 객체 삭제 및 CUDA 캐시 정리 완료


# 3. 학습

In [ ]:
# ───────────────────────────────────────────────────────────
# 학습 루프 (AMP + progress bar + 안전 가드)
# ───────────────────────────────────────────────────────────


EPOCHS     = 5
BATCHES    = len(loader)
CLIP_NORM  = 1.0
NEG_K      = 10          # 음성 ↔ 양성 비율

scaler = GradScaler()
opt    = torch.optim.Adam(model.parameters(), lr=1e-3)
model.train()

for ep in range(1, EPOCHS + 1):
    ep_start = time.perf_counter()
    running  = 0.0

    pbar = tqdm.tqdm(loader, desc=f"Epoch {ep}", ncols=100)

    for batch in pbar:
        batch = batch.to(device, non_blocking=True)

        # ── AMP FP16 forward ────────────────────────────────
        with autocast(device_type='cuda'):
            z   = model(batch.x_dict, batch.edge_index_dict)
            src = z['user'   ][batch['user','buys','product'].edge_label_index[0]]
            dst = z['product'][batch['user','buys','product'].edge_label_index[1]]
            logits = (src * dst).sum(-1)

            loss = torch.nn.functional.binary_cross_entropy_with_logits(
                logits,
                batch['user','buys','product'].edge_label,
                pos_weight=torch.tensor(NEG_K, device=device)
            )

        # ── backward + optimizer step ───────────────────────
        scaler.scale(loss).backward()
        clip_grad_norm_(model.parameters(), CLIP_NORM)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)

        running += loss.item()

        # 진행바에 평균 손실 표시 (첫 배치 대비 0 나눗셈 방지)
        if pbar.n:                           # pbar.n == processed batches
            pbar.set_postfix(loss=f"{running/pbar.n:.4f}")

        # 선택적으로 빈 캐시—VRAM 단편화 방지
        torch.cuda.empty_cache()

    print(f"Epoch {ep} | mean loss={running/BATCHES:.4f} | "
          f"{time.perf_counter()-ep_start:.1f}s")


Epoch 1:  89%|█████████████████████████████████    | 7405/8275 [10:11<01:10, 12.38it/s, loss=0.8183]

# 4. 평가

In [ ]:
# ───────────────────────────────────────────────────────────
# 1. Precision@K 평가
# ───────────────────────────────────────────────────────────
TOP_K = 5
CHUNK = 200_000          # 제품 수가 많을 때 GPU-메모리 절약용

@torch.no_grad()
def precision_at_k(model: nn.Module,
                   te_data: HeteroData,
                   gt_dict: dict[int, list[int]],
                   k: int = TOP_K) -> float:
    """전체 테스트 유저의 평균 Precision@k 계산"""
    model.eval()
    te_data = te_data.to(device)
    z = model(te_data.x_dict, te_data.edge_index_dict)

    user_emb = z['user'].cpu()      # [N_user , D]
    prod_emb = z['product'].cpu()   # [N_prod , D]

    prec = []
    for mid, gt in gt_dict.items():
        if mid not in m2i:                         # 안전: 매핑 안 되면 skip
            continue
        u_vec = user_emb[m2i[mid]]                 # [D]

        # ── 대량 제품일 때 chunk 단위로 top-k ──
        top_scores, top_idx = torch.tensor([]), torch.tensor([])
        for s in range(0, len(prod_emb), CHUNK):
            chunk_score = torch.matmul(prod_emb[s:s+CHUNK], u_vec)
            t_score, t_idx = chunk_score.topk(k)
            top_scores = torch.cat([top_scores, t_score])
            top_idx    = torch.cat([top_idx,    t_idx + s])
        final_idx = top_scores.topk(k).indices
        top_pid   = i2p[top_idx[final_idx].numpy()]

        prec.append(len(set(top_pid) & set(gt)) / k)

    return float(np.mean(prec))

t0 = time.perf_counter()
mean_p5 = precision_at_k(model, test_data, gt_dict, TOP_K)
log(f"Precision@{TOP_K} = {mean_p5:.4f}", t0)



### 저장

In [ ]:
CKPT_PATH = "hetsage_checkpoint.pt"
torch.save({
    "model_state": model.state_dict(),   # HetSAGE 가중치
    "m2i": m2i, "o2i": o2i, "p2i": p2i,  # ID ↔ index 매핑 dict
    "prod_feat_dim": prod_feat.size(1),  # 임베딩 차원(로드시 검증용)
    "train_params": {
        "NEG_K": NEG_K,
        "hid_dim": 64,
        "out_dim": 64,
        "epochs": EPOCHS
    }
}, CKPT_PATH)
print("✔ 체크포인트 저장 완료 :", CKPT_PATH)